# MP1 - Last.fm API with pandas

**Research question #1:** Who are the top 20 artists on Last.fm in the United States, Brazil, and South Korea — and how similar are those charts when we treat **profile country** as geography (including bias)?

**Research question #2:** Which **genres** (operationalized as Last.fm **tags**) have the most **listeners**—measured by each tag’s **`reach`** on the platform?

**API keys are secrets.** Do not paste them into notebooks you commit to GitHub.

Put your key in `.env` next to this notebook:

`LASTFM_API_KEY=your_key_here`

(no quotes, no spaces around `=`). Or run `export LASTFM_API_KEY=...` before Jupyter.


In [16]:
import json
import os
import time
import urllib.parse
import urllib.request
from pathlib import Path

import pandas as pd


def _parse_env_file(path: Path) -> dict:
    """Parse KEY=VALUE lines (works without python-dotenv)."""
    out = {}
    if not path.is_file():
        return out
    for raw in path.read_text(encoding="utf-8").splitlines():
        line = raw.strip()
        if not line or line.startswith("#"):
            continue
        if "=" not in line:
            continue
        key, _, val = line.partition("=")
        key = key.strip()
        val = val.strip().strip('"').strip("'")
        if key:
            out[key] = val
    return out


def _candidate_env_paths() -> list[Path]:
    cwd = Path.cwd()
    paths = [
        cwd / ".env",
        cwd / "Week 5" / "A5: Pandas Assignment" / ".env",
        cwd / "A5: Pandas Assignment" / ".env",
    ]
    for parent in [cwd, *list(cwd.parents)[:10]]:
        paths.append(parent / "Week 5" / "A5: Pandas Assignment" / ".env")
        paths.append(parent / ".env")
    seen: set[Path] = set()
    uniq: list[Path] = []
    for p in paths:
        try:
            rp = p.resolve()
        except OSError:
            continue
        if rp not in seen:
            seen.add(rp)
            uniq.append(p)
    return uniq


def resolve_lastfm_api_key() -> str:
    key = (os.environ.get("LASTFM_API_KEY") or "").strip()
    if key:
        return key
    try:
        from dotenv import load_dotenv

        for p in _candidate_env_paths():
            if p.is_file():
                load_dotenv(p)
                break
        key = (os.environ.get("LASTFM_API_KEY") or "").strip()
        if key:
            return key
    except ImportError:
        pass
    for p in _candidate_env_paths():
        data = _parse_env_file(p)
        key = (data.get("LASTFM_API_KEY") or "").strip()
        if key:
            return key
    raise ValueError(
        "LASTFM_API_KEY is missing or empty. Edit .env in this folder and set:\n"
        "  LASTFM_API_KEY=your_key\n"
        "(no spaces around =). Or: export LASTFM_API_KEY=your_key"
    )


API_KEY = resolve_lastfm_api_key()
BASE = "https://ws.audioscrobbler.com/2.0/"


def lastfm_call(method: str, extra: dict | None = None) -> dict:
    params = {"method": method, "api_key": API_KEY, "format": "json"}
    if extra:
        params.update(extra)
    url = BASE + "?" + urllib.parse.urlencode(params)
    with urllib.request.urlopen(url, timeout=30) as resp:
        payload = json.load(resp)
    err = payload.get("error")
    if err is not None:
        msg = payload.get("message", str(payload))
        raise RuntimeError("Last.fm API error {}: {}".format(err, msg))
    return payload


print("pandas:", pd.__version__)


pandas: 3.0.2


## Research Question 1 — Top artists by profile country

The cells below call **`geo.getTopArtists`** for the United States, Brazil, and South Korea, then compare chart overlap and bias. **Research Question 2** (tags / genres) starts later in this notebook.


## Fetch top artists by country

Uses [`geo.getTopArtists`](https://www.last.fm/api/show/geo.getTopArtists) with `limit=20`. The next cell calls the API for all three countries, then builds **`df`** with columns **`country_label`**, **`name`**, and **`listeners`** only.

Country names must match what Last.fm expects (ISO-style). **Error 6 `country param invalid`** means the string was not recognized; the code tries common alternates (e.g. `Korea, Republic of` for South Korea).

Rankings reflect **Last.fm chart popularity** in that country, not global sales or streams elsewhere.



In [20]:
# I wanted to compare the top artists in the US, Brazil, and South Korea
# I set the parameters for the api call to those three countries and set a limit to 20 artists
COUNTRY_QUERY = [
    # Last.fm expects ISO-style country names; if one fails, we try alternates.
    ("United States", ["United States", "United States of America"]),
    ("Brazil", ["Brazil"]),
    ("South Korea", ["Korea, Republic of", "South Korea"]),
]
LIMIT = 20
parts = []
for label, country_options in COUNTRY_QUERY:
    payload = None
    used_param = None
    for country_param in country_options:
        try:
            payload = lastfm_call(
                "geo.gettopartists",
                {"country": country_param, "limit": LIMIT},
            )
            used_param = country_param
            break
        except RuntimeError as exc:
            err_text = str(exc)
            if "error 6" in err_text and "country param invalid" in err_text:
                continue
            raise
    if payload is None:
        raise RuntimeError(
            "Could not resolve country name for {!r}. Tried: {}".format(
                label, country_options
            )
        )

    artists = payload["topartists"]["artist"]
    if isinstance(artists, dict):
        artists = [artists]
    chunk = pd.json_normalize(artists)
    chunk.insert(0, "country_label", label)
    chunk.insert(1, "country_param", used_param)
    parts.append(chunk)
    time.sleep(0.3)

#I used the concat function to combine the parts into a single dataframe
geo_top_artists = pd.concat(parts, ignore_index=True)

#I defined df to be geo_top_artists to make it easier to code with
#the columns were labeled to be country_label, name, and listeners
df = geo_top_artists.copy()
TABLE_COLS = ["country_label", "name", "listeners"]
TABLE_COLS = [c for c in TABLE_COLS if c in df.columns]
if "listeners" in df.columns:
    df["listeners"] = pd.to_numeric(df["listeners"], errors="coerce")

#this is the last line, showing that the table is the cell output
df[TABLE_COLS]


,country_label,name,listeners
0,United States,Kanye West,242496
1,United States,Drake,237758
2,United States,"Tyler, The Creator",228439
3,United States,Kendrick Lamar,228045
4,United States,PinkPantheress,214837
5,United States,Radiohead,207023
6,United States,The Weeknd,198756
7,United States,Rihanna,197587
8,United States,Tame Impala,191374
9,United States,Frank Ocean,184029


## Countries as columns (side‑by‑side)

Each country still has **20 artists**, so rows are **rank** (1 = top on Last.fm for that country). There are two tables: **artist names** and **listener counts**, both with the same country column headers.

If you need every artist’s name and listeners in **one** grid, merge those two tables into one sheet in Excel or stack them vertically below.


In [25]:
try:
    from IPython.display import display
except ImportError:
    def display(*xs):
        for x in xs:
            print(x)
#I made a copy of df so that I can add columns without affecting the original df
_df = df.copy()
_df["rank"] = _df.groupby("country_label", sort=False).cumcount() + 1
#I used the pivot function to create one table that focused on the top artists of each country

# I wanted to just show the top 10 artists to make the table more concise
# I defined TOP_N to be 10 and used the head function to only show the top 10 artists
TOP_N = 10

names_by_country = _df.pivot(index="rank", columns="country_label", values="name").head(TOP_N)
#I printed the two tables to the console
print("Artist names (columns = country)")
display(names_by_country)




Artist names (columns = country)


country_label,Brazil,South Korea,United States
rank,,,
1,Lady Gaga,Justin Bieber,Kanye West
2,Sabrina Carpenter,Kanye West,Drake
3,Ariana Grande,ILLIT,"Tyler, The Creator"
4,Taylor Swift,The Weeknd,Kendrick Lamar
5,Marina Sena,Kendrick Lamar,PinkPantheress
6,Anitta,BTS,Radiohead
7,Lana Del Rey,LE SSERAFIM,The Weeknd
8,Michael Jackson,Ariana Grande,Rihanna
9,The Weeknd,NewJeans,Tame Impala


In [19]:
try:
    from IPython.display import display
except ImportError:
    display = print

for label in df["country_label"].unique():
    sub = df.loc[df["country_label"].eq(label), TABLE_COLS]
    print()
    print("===", label, "===")
    display(sub.reset_index(drop=True))



=== United States ===


,country_label,name,listeners
0,United States,Kanye West,242926
1,United States,Drake,238213
2,United States,Kendrick Lamar,228488
3,United States,"Tyler, The Creator",228055
4,United States,PinkPantheress,214472
5,United States,Radiohead,206407
6,United States,The Weeknd,198587
7,United States,Rihanna,198126
8,United States,Tame Impala,191412
9,United States,Michael Jackson,186669



=== Brazil ===


,country_label,name,listeners
0,Brazil,Lady Gaga,117912
1,Brazil,Sabrina Carpenter,107347
2,Brazil,Ariana Grande,104454
3,Brazil,Taylor Swift,103022
4,Brazil,Marina Sena,100823
5,Brazil,Anitta,98481
6,Brazil,Lana Del Rey,97000
7,Brazil,Michael Jackson,96004
8,Brazil,The Weeknd,94252
9,Brazil,Rihanna,90366



=== South Korea ===


,country_label,name,listeners
0,South Korea,Justin Bieber,786
1,South Korea,Kanye West,744
2,South Korea,ILLIT,707
3,South Korea,The Weeknd,638
4,South Korea,Kendrick Lamar,636
5,South Korea,BTS,625
6,South Korea,LE SSERAFIM,622
7,South Korea,Ariana Grande,594
8,South Korea,NewJeans,593
9,South Korea,Drake,589


## Follow-up (Research Question 1) — Geographic representativeness on Last.fm

**Research question:** Among Last.fm users who set their **profile country** to the United States, Brazil, or South Korea, how **similar** are the platform’s **top-artist** charts—and what do those patterns suggest about Last.fm’s **geographic bias**?

**Why this section after the fetch?** Part 1 answers the API literally: who ranks on each country’s chart. While doing that, it became clear Last.fm’s **geo** data are **heavily skewed**. `geo.getTopArtists` is driven by **profile country** that users set—not IP address, citizenship, or a representative national sample. In places where Last.fm’s **active userbase is tiny** (e.g. South Korea), the chart can look like **global mainstream** acts (e.g. Justin Bieber at #1) rather than what domestic charts or typical streaming habits in that country would suggest. That pattern fits **a small, non-representative slice** of users—often internationally oriented or tech-savvy listeners who chose that location—not “what everyone in that country listens to.” So Part 1 stays as a **straight API reference** for the professor; **Part 2 adds a comparative lens** (overlap between charts, shared mega-stars across “countries”) to show **platform geography bias** rather than pretending each chart equals national taste.

**How to run this section:** Run the notebook **from the top** through the Part 1 **geo fetch** cell (the one that builds `df`) **before** you run the code cell below. You do **not** need to keep restarting the kernel—only restart if you change your API key in `.env`, switch Python environments, or want a completely fresh session.

**Data caveat:** `geo.getTopArtists` uses **self-reported profile location**, not IP geolocation or national listening surveys.


In [ ]:
try:
    from IPython.display import display
except ImportError:
    display = print

from itertools import combinations

if "df" not in globals():
    raise RuntimeError("Run the Part 1 geo fetch cell first (it creates `df`).")


def jaccard(a: set, b: set) -> float:
    u = a | b
    if not u:
        return 1.0 if not a and not b else 0.0
    return len(a & b) / len(u)


by_country = {
    c: set(df.loc[df["country_label"].eq(c), "name"].astype(str))
    for c in sorted(df["country_label"].unique())
}

print("How many distinct artists per chart:", {c: len(by_country[c]) for c in by_country})

pairs = []
for c1, c2 in combinations(by_country.keys(), 2):
    s1, s2 = by_country[c1], by_country[c2]
    pairs.append(
        {
            "country_a": c1,
            "country_b": c2,
            "overlap": len(s1 & s2),
            "jaccard": round(jaccard(s1, s2), 4),
        }
    )

print("\nPairwise overlap (shared artist names between charts):")
display(pd.DataFrame(pairs))

all_three = set.intersection(*by_country.values())
print(f"\nIn all three charts ({len(all_three)} artists):", sorted(all_three))

unique_summary = []
for c in by_country:
    rest = set.union(*[by_country[o] for o in by_country if o != c]) if len(by_country) > 1 else set()
    only_here = sorted(by_country[c] - rest)
    unique_summary.append(
        {
            "country": c,
            "only_on_this_chart": len(only_here),
            "examples": ", ".join(only_here[:8]) + ("..." if len(only_here) > 8 else ""),
        }
    )

print("\nArtists that show up on only one country's chart:")
display(pd.DataFrame(unique_summary))


## Research Question 2 — Which genres have the most listeners?

**Precise wording for the API:** Which Last.fm **tags** have the highest **`reach`**?

**Definitions:**

- **Genre (here)** means a Last.fm **tag**—community-applied labels that often match genres (rock, hip-hop) but are **not** a strict music-industry taxonomy.
- **Listeners (here)** means the API field **`reach`**: the count of users associated with that tag on Last.fm ([`tag.getInfo`](https://www.last.fm/api/show/tag.getInfo)). **`taggings`** counts how often the tag was applied globally; this notebook ranks by **`reach`** as the closer analogue to “audience size” for a tag.

Data source: **[`chart.getTopTags`](https://www.last.fm/api/show/chart.getTopTags)** (global tag chart).

Run the **setup** cell first so `lastfm_call` and `pandas` are loaded.


In [ ]:
LIMIT_TAGS = 50

payload = lastfm_call("chart.gettoptags", {"limit": LIMIT_TAGS})
tags_raw = payload["tags"]["tag"]
if isinstance(tags_raw, dict):
    tags_raw = [tags_raw]

tags_df = pd.json_normalize(tags_raw)
for col in ("reach", "taggings"):
    if col in tags_df.columns:
        tags_df[col] = pd.to_numeric(tags_df[col], errors="coerce")

tags_df = tags_df.sort_values("reach", ascending=False, na_position="last").reset_index(drop=True)

display_cols = [c for c in ("name", "reach", "taggings") if c in tags_df.columns]
tags_df[display_cols].head(20)


In [ ]:
import matplotlib.pyplot as plt

top_n = 15
plot_df = tags_df.head(top_n).iloc[::-1]  # reverse so highest is at top of horizontal bars

fig, ax = plt.subplots(figsize=(10, max(5, top_n * 0.35)))
ax.barh(plot_df["name"].astype(str), plot_df["reach"])
ax.set_xlabel("reach (Last.fm users)")
ax.set_title(f"Top {top_n} tags by reach — Last.fm chart.getTopTags")
plt.tight_layout()
plt.show()


### Limitations (Research Question 2)

Last.fm tags overlap with genres only loosely. **`reach`** reflects activity **on Last.fm**, not worldwide streams or radio play. Rankings **change over time**. Prefer interpreting results as **platform-specific popularity of tags**, not ground truth for “all listeners everywhere.”
